# MC=0.0 케이스 상세 분석

996개 궤적 중 MC가 0.0인 456개(45.8%)를 하나씩 분석합니다.

In [7]:
import json
from IPython.display import display, HTML
import re

In [8]:
# Load all trajectories
trajectories = []
with open('../outputs/hybrid_1000q_from_0/results_hybrid_all_996.jsonl', 'r') as f:
    for line in f:
        trajectories.append(json.loads(line))

print(f"✓ Loaded {len(trajectories)} trajectories")

✓ Loaded 996 trajectories


In [9]:
# Filter MC=0.0 cases
mc_zero_cases = []

for traj in trajectories:
    if traj['steps']:
        final_mc = traj['steps'][-1]['mc_after']
        if final_mc == 0.0:
            mc_zero_cases.append(traj)

print(f"✓ Found {len(mc_zero_cases)} trajectories with MC=0.0")
print(f"  Percentage: {len(mc_zero_cases)/len(trajectories)*100:.1f}%")

✓ Found 456 trajectories with MC=0.0
  Percentage: 45.8%


In [10]:
# Helper functions for display

def extract_query(content):
    """Extract search query from RAG step content."""
    match = re.search(r'<start_search>(.*?)<end_search>', content)
    if match:
        return match.group(1)
    return None

def truncate(text, max_len=150):
    """Truncate text with ellipsis."""
    if len(text) > max_len:
        return text[:max_len] + "..."
    return text

def get_step_type(step):
    """Get step type from action field."""
    action = step.get('action', 'Unknown')
    if action == 'Search':
        return 'rag'
    elif action == 'Reason':
        return 'cot'
    elif action == 'Finish':
        return 'finish'
    else:
        return 'unknown'

def display_trajectory(traj, index):
    """Display a single trajectory in nice format."""
    
    # Header
    html = f"""
    <div style="border: 2px solid #333; padding: 20px; margin: 20px 0; border-radius: 10px; background-color: #f9f9f9;">
        <h2 style="color: #d00;">📋 Case #{index + 1} / {len(mc_zero_cases)}</h2>
        <p><strong>Question ID:</strong> {traj['question_id']}</p>
        <p><strong>Correct:</strong> {'✅ YES' if traj['is_correct'] else '❌ NO'}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #fff;">
        <h3>🔍 Question</h3>
        <p style="font-size: 16px; line-height: 1.6;">{traj['question']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #e8f5e9;">
        <h3>✅ Gold Answer</h3>
        <p style="font-size: 16px; font-weight: bold;">{traj['gold_answer']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: {'#ffebee' if not traj['is_correct'] else '#e8f5e9'};">
        <h3>🤖 Predicted Answer</h3>
        <p style="font-size: 16px;">{traj['predicted_answer']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #fff3e0;">
        <h3>📊 Summary</h3>
        <p><strong>Total Steps:</strong> {traj['num_steps']} (CoT: {traj['num_cot_steps']}, RAG: {traj['num_rag_steps']})</p>
        <p><strong>Has RAG:</strong> {'Yes' if traj['has_rag'] else 'No'}</p>
        <p><strong>Final MC:</strong> <span style="color: red; font-weight: bold;">{traj['steps'][-1]['mc_after']:.3f}</span></p>
    </div>
    """
    
    # Steps
    html += '<div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #f5f5f5;">'
    html += '<h3>🔄 Step-by-Step Trajectory</h3>'
    
    for step in traj['steps']:
        step_type = get_step_type(step)
        step_color = '#e3f2fd' if step_type == 'rag' else '#fff9c4' if step_type == 'cot' else '#f0f0f0'
        
        html += f'<div style="border-left: 4px solid #2196F3; padding: 10px; margin: 10px 0; background-color: {step_color};">'
        html += f'<h4>Step {step["step_num"]} - {step.get("action", "Unknown").upper()}</h4>'
        html += f'<p><strong>MC:</strong> {step["mc_before"]:.3f} → {step["mc_after"]:.3f} '
        html += f'(Δ {step["mc_after"] - step["mc_before"]:+.3f})</p>'
        html += f'<p><strong>RPE:</strong> {step["rpe"]:.3f} | <strong>Label:</strong> {step["label"]}</p>'
        
        if step_type == 'rag':
            query = step.get('action_input') or extract_query(step['content'])
            if query:
                html += f'<p><strong>🔍 Query:</strong> <code>{query}</code></p>'
            
            # Show observation and sub_answer for RAG
            if step.get('observation'):
                obs_preview = truncate(str(step['observation']), 150)
                html += f'<p><strong>📄 Observation:</strong> {obs_preview}</p>'
            if step.get('sub_answer'):
                sub_preview = truncate(str(step['sub_answer']), 150)
                html += f'<p><strong>💡 Sub-answer:</strong> {sub_preview}</p>'
        
        # Show content preview
        content_preview = step['content'].replace('\n', ' ')[:300]
        html += f'<details><summary><strong>💬 Content (click to expand)</strong></summary>'
        html += f'<pre style="white-space: pre-wrap; background-color: #fff; padding: 10px; border-radius: 5px;">{step["content"]}</pre>'
        html += f'</details>'
        
        html += '</div>'
    
    html += '</div>'
    
    display(HTML(html))

## 분석 시작

아래 셀을 실행하면 한 케이스씩 볼 수 있습니다. `current_index`를 변경해서 다른 케이스를 확인하세요.

In [11]:
# 현재 볼 케이스 번호 (0부터 시작)
current_index = 0

if current_index < len(mc_zero_cases):
    display_trajectory(mc_zero_cases[current_index], current_index)
else:
    print(f"❌ Index {current_index} out of range. Max: {len(mc_zero_cases) - 1}")

## 빠른 네비게이션

여러 케이스를 빠르게 보려면 아래 셀을 사용하세요.

In [12]:
# 다음 케이스
current_index += 1

if current_index < len(mc_zero_cases):
    display_trajectory(mc_zero_cases[current_index], current_index)
else:
    print(f"✓ 마지막 케이스입니다! (총 {len(mc_zero_cases)}개)")

In [13]:
# 이전 케이스
current_index -= 1

if current_index >= 0:
    display_trajectory(mc_zero_cases[current_index], current_index)
else:
    print(f"✓ 첫 번째 케이스입니다!")

In [ ]:
# 특정 번호로 이동
current_index = 10  # 원하는 번호로 변경

if 0 <= current_index < len(mc_zero_cases):
    display_trajectory(mc_zero_cases[current_index], current_index)
else:
    print(f"❌ Index {current_index} out of range (0 ~ {len(mc_zero_cases) - 1})")

## 통계 분석

MC=0.0 케이스들의 패턴을 찾아봅니다.

In [ ]:
# MC=0.0 케이스들의 정답률
correct_count = sum(1 for traj in mc_zero_cases if traj['is_correct'])
incorrect_count = len(mc_zero_cases) - correct_count

print("MC=0.0 케이스 통계")
print("=" * 50)
print(f"총 {len(mc_zero_cases)}개")
print(f"✅ 정답: {correct_count} ({correct_count/len(mc_zero_cases)*100:.1f}%)")
print(f"❌ 오답: {incorrect_count} ({incorrect_count/len(mc_zero_cases)*100:.1f}%)")
print()

# RAG 사용 여부
with_rag = sum(1 for traj in mc_zero_cases if traj['has_rag'])
cot_only = len(mc_zero_cases) - with_rag

print(f"RAG 사용: {with_rag} ({with_rag/len(mc_zero_cases)*100:.1f}%)")
print(f"CoT Only: {cot_only} ({cot_only/len(mc_zero_cases)*100:.1f}%)")
print()

# 평균 단계 수
avg_steps = sum(traj['num_steps'] for traj in mc_zero_cases) / len(mc_zero_cases)
avg_cot_steps = sum(traj['num_cot_steps'] for traj in mc_zero_cases) / len(mc_zero_cases)
avg_rag_steps = sum(traj['num_rag_steps'] for traj in mc_zero_cases) / len(mc_zero_cases)

print(f"평균 단계: {avg_steps:.1f}")
print(f"  CoT: {avg_cot_steps:.1f}")
print(f"  RAG: {avg_rag_steps:.1f}")

In [ ]:
# MC가 항상 0.0이었는지 확인
always_zero = 0
became_zero = 0

for traj in mc_zero_cases:
    all_zero = all(step['mc_after'] == 0.0 for step in traj['steps'])
    
    if all_zero:
        always_zero += 1
    else:
        became_zero += 1

print("MC 변화 패턴")
print("=" * 50)
print(f"항상 0.0: {always_zero} ({always_zero/len(mc_zero_cases)*100:.1f}%)")
print(f"중간에 0.0이 됨: {became_zero} ({became_zero/len(mc_zero_cases)*100:.1f}%)")

## 정답이지만 MC=0.0인 케이스

이 케이스들을 분석하면 MC 계산 로직의 문제를 찾을 수 있습니다.

In [ ]:
# 정답이지만 MC=0.0
correct_but_mc_zero = [traj for traj in mc_zero_cases if traj['is_correct']]

print(f"정답이지만 MC=0.0: {len(correct_but_mc_zero)}개")
print("="* 50)

for i, traj in enumerate(correct_but_mc_zero[:5]):  # 처음 5개만
    print(f"\n{i+1}. {traj['question_id']}")
    print(f"   Question: {truncate(traj['question'], 80)}")
    print(f"   Gold: {traj['gold_answer']}")
    print(f"   Predicted: {truncate(traj['predicted_answer'], 80)}")
    print(f"   Steps: {traj['num_steps']} (CoT: {traj['num_cot_steps']}, RAG: {traj['num_rag_steps']})")

## 특정 케이스 상세 분석

정답인데 MC=0.0인 첫 번째 케이스를 자세히 봅니다.

In [ ]:
if correct_but_mc_zero:
    display_trajectory(correct_but_mc_zero[0], 0)
else:
    print("정답인데 MC=0.0인 케이스가 없습니다.")

## Export 분석 결과

분석한 케이스들을 파일로 저장합니다.

In [ ]:
# MC=0.0 케이스만 별도 저장
output_file = '../outputs/hybrid_1000q_from_0/mc_zero_cases.jsonl'

with open(output_file, 'w') as f:
    for traj in mc_zero_cases:
        f.write(json.dumps(traj) + '\n')

print(f"✓ Saved {len(mc_zero_cases)} MC=0.0 cases to: {output_file}")